# Chinese Character Database — Analysis Notebook

**Author:** Robert W Cellucci  
**Project:** CJK (Chinese-Japanese-Korean) SQLite Character Database  

This notebook queries the pre-built `Chinese_Character_Database.db` and is **read-only** with respect to the database. The companion creation notebook builds the database from raw source files.

Every query in this notebook opens and closes a connection via a `with` block, enforces foreign key constraints with `PRAGMA foreign_keys = ON`, and reads results directly into a pandas DataFrame. Nothing is written back to disk.

## Analytical Narrative

Chinese, Japanese, and Korean each maintain their own official character standards
for education and literacy. Yet all three draw from the same logographic pool. A
Japanese learner working through the Jōyō list, a Mandarin learner working through
HSK 3.0, a Traditional Chinese learner working through Taiwan's MoE standard, and a
Korean learner working through the Korean Educational Hanja list are all engaging with
largely the same script with no published map of where those curricula actually overlap.

This notebook works toward one central finding: **which characters represent a
cross-validated, multi-institutional consensus on foundational Chinese character literacy?**

The sections below build toward that answer incrementally. Sections 1 and 2 characterize
Japan's lists in isolation, first identifying what Japan considers essential that no other
system shares, then tracing how Japan's traditional-form kanji relate to Mainland China's
simplification reforms. Sections 3 and 4 establish pairwise overlaps between Japan and its
East Asian neighbors. Section 5 delivers the core finding: the strict three-way intersection
of Japan, Taiwan, and Korea — **2,573 characters** independently ratified by three national
educational authorities.

## Section Map

| Section | Question | Source Lists Involved |
|---|---|---|
| **1** | Japan Exclusivity: which Japanese characters appear in no other regional list? | `Japan` |
| **2** | How characters from Japan's lists were simplified in China's script reforms | `Japan`, `trad_simp_map` |
| **3** | Japan ↔ Korea pairwise overlap | `Japan`, `Korea` |
| **4** | Japan ↔ Taiwan pairwise overlap | `Japan`, `Taiwan` |
| **5** | Pan-CJK three-way intersection: core finding | `Japan`, `Taiwan`, `Korea` |

---

## Import Statements

Only two top-level imports are needed:

- `pandas` — in-memory data wrangling and the staging layer for all query results
- `sqlite3` — Python's built-in SQLite driver; no installation required

In [2]:
import pandas as pd           # DataFrame-based query result handling and display
import sqlite3 as sql         # CPython built-in SQLite driver; no pip install needed

## Database Validation

Before running any analytical queries, it is worth confirming that the database is
structurally sound: all expected tables are present, row counts are plausible, and
foreign key relationships are intact. This section is a sanity check, not part of
the analysis proper but it catches silent corruption or a stale `.db` file early,
before a bad join produces a misleadingly empty result downstream.

The checks below query `sqlite_master` for the table manifest, then pull row counts
from each core table. Expected counts are noted inline for reference.

In [15]:
# ── Database Validation ────────────────────────────────────────────────────────
#       Expected tables:
#         - character_table      (spine of the schema; one row per unique codepoint)
#         - character_source     (list membership: Joyo, Jinmeiyo, MoE, HSK, Hanja, etc.)
#         - dic_index_table  (entry numbers in KangXi, Morohashi, Nelson, etc.)
#         - trad_simp_map        (3,165 Traditional → Simplified codepoint pairs)

with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    master_table = pd.read_sql_query("""
    SELECT *
    FROM sqlite_master
    """,conn)
display(master_table)

,type,name,tbl_name,rootpage,sql
0,table,character_table,character_table,2,CREATE TABLE character_table (\n codepo...
1,table,character_source,character_source,3,CREATE TABLE character_source (\n c...
2,index,sqlite_autoindex_character_source_1,character_source,5,None
3,table,dic_index_table,dic_index_table,4,CREATE TABLE dic_index_table (\n co...
4,table,trad_simp_map,trad_simp_map,6,CREATE TABLE trad_simp_map (\n trad_cod...
5,index,sqlite_autoindex_trad_simp_map_1,trad_simp_map,15,None


In [18]:
# TODO: Pull row counts from character_table and character_source.
#       Use these as a baseline — unexpected counts indicate a bad load or re-run issue.
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    display(pd.read_sql_query("""
    SELECT *
    FROM character_table
    """,conn))

with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    display(pd.read_sql_query("""
    SELECT *
    FROM character_source
    """,conn))

,codepoint,literal
0,13466,㒚
1,13527,㓗
2,13589,㔕
3,13601,㔡
4,13823,㗿
...,...,...
9835,173668,𪙤
9836,180501,𬄕
9837,182227,𬟓
9838,189801,𮕩


,codepoint,source,source_region
0,30340,hsk_trad,China
1,20102,hsk_trad,China
2,30637,hsk_trad,China
3,22312,hsk_trad,China
4,26159,hsk_trad,China
...,...,...,...
28166,40860,KoreanName,Korea
28167,40861,KoreanName,Korea
28168,40864,KoreanName,Korea
28169,40866,KoreanName,Korea


In [31]:
# TODO: Run PRAGMA foreign_key_check to confirm referential integrity is intact.
#       An empty result is the expected (passing) output.
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    result = conn.execute("PRAGMA foreign_key_check;").fetchall()
    
display(result)

[]

---

## The Japanese Primary Kanji List

Japan maintains two official kanji lists administered by the Ministry of Education:

- **Jōyō kanji** (常用漢字) — 2,136 characters designated for general use in newspapers,
  government documents, and public communication. These are the characters every student
  is expected to master by the end of compulsory education.
- **Jinmeiyō kanji** (人名用漢字) — 863 characters approved for use in personal names
  but not required for general literacy. They extend the usable character set for naming
  purposes beyond the Jōyō core.

Together these two lists define the full scope of officially sanctioned kanji in Japan.
The query below retrieves all 3,003 characters across both lists, which will serve as
the Japan-side operand for all subsequent comparisons.

In [3]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")   # SQLite disables FK enforcement by default; this reinstates it
    japanese_characters = pd.read_sql_query("""
    SELECT 
        cs.codepoint,     -- Unicode scalar value; the primary key used in all joins
        ct.literal        -- The actual character glyph, derived from chr(codepoint)
    FROM character_source AS cs
    JOIN character_table AS ct
        ON cs.codepoint = ct.codepoint
    WHERE (cs.source = 'Joyo' OR cs.source = 'Jinmeiyo')  -- Both official Japanese lists
    """, conn)

display(japanese_characters)   # Expected: 3,003 rows

,codepoint,literal
0,19968,一
1,19969,丁
2,19971,七
3,19975,万
4,19976,丈
...,...,...
2998,64101,贈
2999,64103,逸
3000,64104,難
3001,64105,響


### Characters Exclusive to Japan

Of Japan's 3,003 kanji, how many appear on no other regional educational list?
The query below uses a `NOT EXISTS` correlated subquery to filter to codepoints
that have a source row for Japan but no source row for any other region. These
are characters that Japan's curriculum considers essential but that Taiwan's MoE,
Korea's Educational Hanja list, and China's HSK 3.0 all omit.

The result is **268 characters** that includes a significant proportion of Jinmeiyō
kanji (name-use characters with limited general circulation) and a smaller number
of Jōyō kanji that have no clear cognate in other CJK educational frameworks.
A natural next step, implemented in Section 1b below, is to identify which of these
268 are true kokuji characters invented in Japan with no Chinese origin at all.

In [4]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    japan_exclusive = pd.read_sql_query("""
        SELECT
            cs.codepoint,
            ct.literal
        FROM character_source AS cs
        JOIN character_table AS ct
            ON cs.codepoint = ct.codepoint
        WHERE (cs.source = 'Joyo' OR cs.source = 'Jinmeiyo')
        AND NOT EXISTS (
              -- Exclude any codepoint that has a source row outside Japan
              SELECT 1
              FROM character_source AS cs2
              WHERE cs2.codepoint = cs.codepoint
                AND cs2.source_region != 'Japan'
          )
        ORDER BY cs.codepoint
    """, conn)

display(japan_exclusive)   # Expected: 268 rows

,codepoint,literal
0,20001,両
1,20028,丼
2,20055,乗
3,20096,亀
4,20175,仏
...,...,...
263,64101,贈
264,64103,逸
265,64104,難
266,64105,響


### A Note About Kokuji

**Kokuji** (国字, literally "country characters") are kanji invented in Japan. These characters
that have no Chinese etymological origin and exist only in the Japanese writing system.
Well-known examples include 働 (*hataraku*, to work) and 峠 (*tōge*, mountain pass).

Because kokuji were never part of the Chinese character pool, they are by definition
absent from Taiwan's MoE list, Korea's Hanja list, and China's HSK. This makes them a
structural subset of the Japan-exclusive set identified in Section 1a. 

Note: Not all the characters that are unique to japan here are kokuji, by wikipedia's estimate there should only be 16 kokuji in japan's national lists.

## How Japan's Kanji Were Simplified in Mainland China

Japan and Mainland China both reformed their official character sets in the twentieth
century, independently simplifying stroke structures that had remained stable for
centuries. The reforms took different forms including Japan's Jōyō revisions (1946, 1981, 2010)
produced *shinjitai* (新字体) variants; China's reforms (1956, 1964) produced *simplified
characters* (简体字) and the two systems did not coordinate. The result is a complex
mapping where some Japanese and Chinese forms converge on the same simplified shape,
others diverge, and some characters were simplified in only one system.

The `trad_simp_map` table records 3,165 explicit Traditional-to-Simplified codepoint
pairs, derived from the HSK 3.0 dataset. The query below joins Japan's Jōyō list
against this map to surface the **463 Jōyō kanji** that have a recorded Mainland
Chinese simplified counterpart displaying both the traditional form (as used in
Japan) and the simplified form (as used in the PRC) side by side.

Note that this query is scoped to Jōyō only (not Jinmeiyō), since Jinmeiyō characters
are unlikely to appear in HSK-derived simplification data.

In [5]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    j_simp = pd.read_sql_query("""
    SELECT 
        map.trad_codepoint,   -- Codepoint of the traditional form (as used in Japan)
        map.trad_literal,     -- Traditional character glyph
        map.simp_codepoint,   -- Codepoint of the Mainland Chinese simplified form
        map.simp_literal      -- Simplified character glyph
    FROM character_source AS cs
        JOIN trad_simp_map AS map
        ON cs.codepoint = map.trad_codepoint   -- Match Joyo codepoints to their traditional-form entry in the map
    WHERE cs.source = 'Joyo'                   -- Scoped to Joyo only; Jinmeiyo excluded
    ORDER BY cs.codepoint
    """, conn)

display(j_simp)   # Expected: 463 rows

,trad_codepoint,trad_literal,simp_codepoint,simp_literal
0,20006,並,24182,并
1,20094,乾,24178,干
2,20341,併,24182,并
3,20406,侶,20387,侣
4,20418,係,31995,系
...,...,...,...,...
458,39854,鮮,40092,鲜
459,40165,鳥,40479,鸟
460,40180,鳴,40483,鸣
461,40372,鶴,40548,鹤


## Japan ↔ Korea Pairwise Overlap

Korea's Educational Hanja list (한문 교육용 기초 한자) designates 1,800 characters for
instruction in middle and high school. Like Japan's kanji system, Korean Hanja are
traditional-form characters, the same script pool that predates both Japan's shinjitai
reforms and China's simplification program. This shared traditional-form heritage makes
Japan-Korea overlap comparisons methodologically cleaner than any comparison involving
China's simplified forms.

The query below identifies all codepoints that appear in at least one Japanese source
(Jōyō or Jinmeiyō) **and** in the Korean Educational Hanja list. The `HAVING` clause
enforces that both regions are represented, not just that a character appears in some
source from each region, but that both `source_region` values are present for that
codepoint. The `source_count` column reflects the number of *individual lists* (not
regions) a character belongs to, and will exceed 2 for characters present in both
Jōyō and Jinmeiyō simultaneously.

Result: **2,904 characters** which is the largest pairwise overlap between any two individual
national lists in this dataset.

In [6]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    j_k_full = pd.read_sql_query("""
        SELECT
            cs.codepoint,
            ct.literal,
            COUNT(DISTINCT cs.source) AS source_count   -- Number of individual lists (Joyo, Jinmeiyo, Hanja, etc.)
        FROM character_source AS cs
        JOIN character_table AS ct
            ON cs.codepoint = ct.codepoint
        WHERE cs.source_region IN ('Korea', 'Japan')    -- Filter to only Japanese and Korean source rows
        GROUP BY cs.codepoint, ct.literal
        HAVING COUNT(DISTINCT cs.source_region) = 2    -- Strict intersection: both regions must be present
        ORDER BY cs.codepoint
    """, conn)

display(j_k_full)   # Expected: 2,904 rows

,codepoint,literal,source_count
0,19968,一,4
1,19969,丁,4
2,19971,七,4
3,19975,万,4
4,19976,丈,4
...,...,...,...
2899,40766,鼾,3
2900,40778,齊,2
2901,40799,齟,2
2902,40812,齬,2


## Japan ↔ Taiwan Pairwise Overlap

Taiwan's Ministry of Education list (國語常用字表) designates 4,808 characters for
standard Mandarin literacy instruction. Like Korea's Hanja list, it operates entirely
in traditional forms, the script Japan also uses, prior to any shinjitai substitutions.
The Taiwan list is the largest of the four regional lists in this database, which
makes it a generous comparator: nearly any character Japan teaches has a reasonable
chance of appearing in Taiwan's broader inventory.

The query structure is identical to Section 3, with `source_region` filtered to
`Japan` and `Taiwan` instead. The `HAVING COUNT(DISTINCT cs.source_region) = 2`
clause again enforces strict bilateral membership.

Result: **2,599 characters** shared between Japan's combined kanji lists and Taiwan's
MoE standard, a slightly smaller overlap than Japan-Korea, despite Taiwan's list
being considerably larger. This reflects the influence of Japan's shinjitai variants,
which introduced codepoint-level divergences from the traditional forms Taiwan continues
to use.

In [8]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    j_t_full = pd.read_sql_query("""
        SELECT
            cs.codepoint,
            ct.literal,
            COUNT(DISTINCT cs.source) AS source_count   
        FROM character_source AS cs
        JOIN character_table AS ct
            ON cs.codepoint = ct.codepoint
        WHERE cs.source_region IN ('Taiwan', 'Japan')  
        GROUP BY cs.codepoint, ct.literal
        HAVING COUNT(DISTINCT cs.source_region) = 2   
        ORDER BY cs.codepoint
    """, conn)

display(j_t_full)   # Expected: 2,599 rows

,codepoint,literal,source_count
0,19968,一,4
1,19969,丁,4
2,19971,七,4
3,19976,丈,4
4,19977,三,4
...,...,...,...
2594,40766,鼾,3
2595,40778,齊,2
2596,40799,齟,2
2597,40812,齬,2


## Core Finding: The Pan-CJK Three-Way Intersection

The query below is the analytical destination of the entire notebook. It asks a
single question: **which characters appear on all three of Japan's Jōyō list,
Taiwan's Ministry of Education standard, and Korea's Educational Hanja list
simultaneously?**

The answer is **2,573 characters**. This is a set that has been independently ratified
by three distinct national educational authorities, across three languages, over
decades of curriculum revision. No single institution designed this overlap; it
emerged from convergent pedagogical judgment about what logographic literacy
requires.

**Why these three lists, and not China (HSK)?**  
HSK 3.0 is represented in this database in its traditional character forms
(`hsk_trad`), not the simplified forms actually used in PRC classrooms. Including
it in the intersection would compare traditional-form proxies against the genuine
traditional-script standards of Japan, Taiwan, and Korea which would be a methodologically
unsound equivalence. China's data is used in cross-regional comparisons where the
comparison is directionally informative, but it is excluded from the core
intersection for this reason. The 2,573-character set therefore represents the
consensus of the three living traditional-character educational systems, untainted
by the simplification asymmetry.

The `source_count = 3` column confirms every returned row genuinely satisfies all
three membership conditions, this is a strict intersection, not a union or
majority vote.

In [10]:
with sql.connect('Chinese_Character_Database.db') as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    j_t_k_full = pd.read_sql_query("""
        SELECT
            cs.codepoint,
            ct.literal,
            COUNT(DISTINCT cs.source_region) AS source_count   
        FROM character_source AS cs
        JOIN character_table AS ct
            ON cs.codepoint = ct.codepoint
        WHERE cs.source_region IN ('Korea', 'Taiwan', 'Japan')  
        GROUP BY cs.codepoint, ct.literal
        HAVING COUNT(DISTINCT cs.source_region) = 3            
        ORDER BY cs.codepoint
    """, conn)
    
display(j_t_k_full)   # Expected: 2,573 rows; source_count = 3 for every row

,codepoint,literal,source_count
0,19968,一,3
1,19969,丁,3
2,19971,七,3
3,19976,丈,3
4,19977,三,3
...,...,...,...
2568,40766,鼾,3
2569,40778,齊,3
2570,40799,齟,3
2571,40812,齬,3


### Known Limitations

**Han Unification overcounts true overlap.** Unicode merged visually similar CJK
characters into single codepoints. Characters that are functionally distinct across
regions but share a codepoint due to glyph unification are counted as one, inflating
apparent overlap between scripts. Regional glyph variation and simplification schemes
are not captured by the Unihan data alone.

**HSK is represented in traditional forms only.** The `hsk_trad` source contains
traditional written forms of HSK 3.0 vocabulary, not the simplified forms used in
PRC classrooms. Cross-script comparisons involving China reflect traditional
character inventories, which are no longer the characters Mainland learners actually study.